In [1]:
import numpy as np
import pyvista as pv


def plot_data(
        data: np.ndarray | list,
        size: float = 5.0,
):
    coords = data
    colors = np.clip(data, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_clustered_pv(
        data: np.ndarray,
        labels: np.ndarray,
        cluster_centers: np.ndarray,
        size: float = 4.0
):
    coords = data
    cluster_colors = cluster_centers[labels]
    colors = np.clip(cluster_colors, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_centroids_pv(
        cluster_centers: np.ndarray,
        size: float = 18.0
):
    coords = cluster_centers
    colors = np.clip(cluster_centers, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size,
        render_points_as_spheres=True
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()

In [2]:
import numpy as np

def generate_dataset(centers, stds, n_samples=500, ndim=3):
    X = []
    y = []

    for i, center in enumerate(centers):
        cluster = np.random.normal(loc=center, scale=stds[i], size=(n_samples // len(centers), ndim))
        X.append(cluster)
        y.append(np.full(n_samples // len(centers), i))

    return np.vstack(X), np.concatenate(y)

In [3]:
import pandas as pd
import pyvista as pv
import matplotlib.pyplot as plt

default_colors = np.array(plt.colormaps.get_cmap('tab10').colors)

In [4]:
data, ground_truth = generate_dataset(
    [[1, 1, 1], [3, 3, 3], [2, 2, 1]],
    stds=[.5, .5, .5],
    n_samples=500
)

pd.DataFrame(data)

,0,1,2
0,1.193724,0.960182,0.973457
1,0.936950,1.444936,0.795443
2,1.104741,0.430668,0.978692
3,0.179348,1.180967,1.336819
4,0.472643,1.222365,0.380927
...,...,...,...
493,1.807379,1.690230,0.996261
494,1.913751,2.356095,1.467560
495,1.946545,1.994458,1.192393
496,1.933946,1.528952,0.952119


In [5]:
plt = pv.Plotter()

for n, d in zip(ground_truth, data):
    cloud = pv.PolyData(d)
    color = default_colors[n % len(default_colors)]

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=10
    )

plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42417/index.html?ui=P_0x7f2e71589940_0&reconnect=auto" class="pyvi…

In [6]:
from ktree.ntree import NTreeDynamic

tree = NTreeDynamic(1)

for a in data:
    tree.insert(a)

sorted_data = tree.sort()

In [7]:
plt = pv.Plotter()

for n, cluster in enumerate(sorted_data):
    s_data = list(cluster)
    cloud = pv.PolyData(s_data)
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=2
    )

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=len(s_data) // 2
    )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42417/index.html?ui=P_0x7f2e6a680050_1&reconnect=auto" class="pyvi…

In [8]:
n_clusters = 2

all_clusters = sorted(sorted_data, key=lambda x: len(x))[::-1]
clusters = all_clusters[:n_clusters]
all_data = all_clusters[n_clusters:]

main_clusters = []

for cluster in clusters:
    c_centroid = np.mean([*cluster], axis=0)
    sum_dist = 0

    sub_cluster = []

    for data in all_data:
        d_centroid = np.mean([*data], axis=0)
        dist = np.linalg.norm(c_centroid - d_centroid)

        sum_dist += dist

        sub_cluster.append((dist, data))

    mead_dist = sum_dist / len(all_data)
    sub_cluster = sorted(sub_cluster, key=lambda a: a[0])

    main_clusters.append((cluster, [data for (dist, data) in sub_cluster if dist < mead_dist]))


for (cluster, all_data) in main_clusters:
    print("Cluster: ", cluster)
    print("Data: ", all_data)


Cluster:  Cluster(axis=[[-0.08282195453576602, 2.069417678486026], [-0.497216983488449, 1.971820522254181], [-0.5209182123391591, 1.816085075850999]])
Data:  [Cluster(axis=[[1.0024349074996926, 2.001493513799422], [0.5940062454783236, 1.836293331589799], [1.9786816974976258, 2.2110441839547947]]), Cluster(axis=[[0.6259381319320129, 2.066159408027743], [1.975621105368175, 3.232760531021266], [-0.06750127876488987, 1.703842828177851]]), Cluster(axis=[[2.0875350114697646, 3.6450452476151955], [-0.09578361547721448, 1.960655780700877], [-0.10024611359092828, 1.713821271723162]])]
Cluster:  Cluster(axis=[[2.1007591969506754, 4.230413230561769], [2.2198745297623113, 4.446307901484895], [1.8709953823451446, 4.186500652827769]])
Data:  [Cluster(axis=[[2.1619329479587828, 3.509587925138516], [1.4193043755851833, 1.9378736377903747], [1.887536366093728, 3.5481037635867945]]), Cluster(axis=[[1.1783933769409582, 2.073216578352416], [2.1991388279013893, 3.301440447308145], [1.8356747581768498, 2.88

In [13]:
plt = pv.Plotter()

for n, (cluster, all_data) in enumerate(main_clusters):
    s_data = list(cluster)

    cloud = pv.PolyData(list(cluster))
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=15
    )

    for data in (all_data + [cluster]):
        cloud = pv.PolyData(list(data))

        plt.add_points(
            cloud,
            color=color,
            render_points_as_spheres=True,
            smooth_shading=True,
            point_size=5
        )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42417/index.html?ui=P_0x7f2e68f8cb90_6&reconnect=auto" class="pyvi…